# Task 1: رحلة داتا السكري

## 1. فهم المشكلة
* **السؤال:** ما هي العوامل المؤثرة والمرتبطة بإصابة المرأة بمرض السكري؟
* **الـ Target:** عمود `Outcome` (حيث `1` تعني مصابة، و`0` تعني غير مصابة).
* **المستفيد:** عيادة السكري، لتحديد الفئات الأكثر عرضة للإصابة والتركيز عليها في برامج الفحص المبكر.

## 2. أول نظرة (استكشاف البيانات الأولي)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# تحميل البيانات
df = pd.read_csv('diabetes.csv')

print("حجم البيانات:", df.shape)
print("\nمعلومات الأعمدة والأنواع:")
print(df.info())
print("\nعدد الصفوف المكررة:", df.duplicated().sum())
df.describe()

**ملاحظاتي من الفحص الأولي:**
* البيانات تحتوي على 768 صف و 9 أعمدة.
* جميع الأعمدة أرقام (int و float) ولا يوجد أي قيم فارغة (Null) ظاهرة، ولا يوجد صفوف مكررة.
* عند النظر للإحصائيات (الحد الأدنى min)، لاحظت وجود أصفار في أعمدة غير منطقية طبياً مثل `Glucose` و `BloodPressure` و `SkinThickness` و `Insulin` و `BMI`.

## 3. التحليل الاستكشافي بالرسومات (EDA)

In [ ]:
# 1. رسم توزيع الـ Target
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Outcome')
plt.title('Outcome Distribution (0 = No Diabetes, 1 = Diabetes)')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.show()

print("النسب المئوية للـ Outcome:")
print((df['Outcome'].value_counts(normalize=True) * 100).round(1))

**التعليق على الرسمة:**
يظهر الرسم أن البيانات غير متوازنة؛ حيث تشكل غير المصابات (0) نسبة 65.1% (500 حالة)، بينما المصابات (1) يشكلن 34.9% (268 حالة).

In [ ]:
# 2. رسم علاقة الجلوكوز ومؤشر كتلة الجسم بالنتيجة
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x='Outcome', y='Glucose', ax=axes[0])
axes[0].set_title('Glucose Level by Outcome')

sns.boxplot(data=df, x='Outcome', y='BMI', ax=axes[1])
axes[1].set_title('BMI by Outcome')

plt.tight_layout()
plt.show()

print("وسيط الجلوكوز لكل فئة:")
print(df.groupby('Outcome')['Glucose'].median())
print("\nوسيط BMI لكل فئة:")
print(df.groupby('Outcome')['BMI'].median())

**التعليق على الرسمة:**
توضح الرسومات أن وسيط الجلوكوز لدى المصابات (140) أعلى بشكل واضح مقارنة بغير المصابات (107)، وكذلك وسيط مؤشر كتلة الجسم أعلى لدى المصابات (34.25) مقارنة بغير المصابات (30.05).

## 4. التنظيف (Data Cleaning) + Decision Log

In [ ]:
# فحص عدد الأصفار في الأعمدة الطبية
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    zeros = (df[col] == 0).sum()
    print(f"{col}: {zeros} صفر ({round(zeros/len(df)*100, 1)}%)")

In [ ]:
# تطبيق قرارات التنظيف
df_clean = df.copy()

# 1. استبدال الأصفار بـ NaN في الأعمدة ذات النسب القليلة وتعويضها بالوسيط (Median)
cols_to_impute = ['Glucose', 'BloodPressure', 'BMI']
for col in cols_to_impute:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 2. التعامل مع الأنسولين وسمك الجلد (نسبة الأصفار عالية جداً)
df_clean['Has_Insulin_Test'] = np.where(df_clean['Insulin'] > 0, 1, 0)
df_clean = df_clean.drop(columns=['Insulin', 'SkinThickness'])

print("تم التنظيف بنجاح. القيم الفارغة المتبقية:", df_clean.isnull().sum().sum())

**سجل القرارات (Decision Log):**
1. **عمود `Glucose`:** تم رصد 5 صفوف بقيمة 0 (0.7%). تم استبدالها بالوسيط (Median) لأن العدد قليل جداً ولا يؤثر على التوزيع.
2. **عمود `BloodPressure`:** تم رصد 35 صف بقيمة 0 (4.6%). تم استبدالها بالوسيط للحفاظ على الصفوف.
3. **عمود `BMI`:** تم رصد 11 صف بقيمة 0 (1.4%). تم استبدالها بالوسيط لأن القيمة صفر مستحيلة طبياً.
4. **عمود `Insulin`:** يحتوي على 374 صفر (48.7% من البيانات). تعويض نصف العمود سيشوه البيانات، لذا استخرجت منه معلومة جديدة (هل تم إجراء فحص أنسولين أم لا `Has_Insulin_Test`) ثم حذفت العمود الأصلي.
5. **عمود `SkinThickness`:** يحتوي على 227 صفر (29.6%). تم حذفه لكثرة القيم المفقودة وضعف ارتباطه مقارنة بباقي الأعمدة.

## 5. هندسة الميزات (Feature Engineering)

In [ ]:
# إنشاء أعمدة جديدة مفيدة طبياً
# 1. تصنيف مؤشر كتلة الجسم (سمنة أم لا)
df_clean['Is_Obese'] = np.where(df_clean['BMI'] >= 30, 1, 0)

# 2. تصنيف مستوى الجلوكوز (مرتفع >= 126 / طبيعي < 126)
df_clean['High_Glucose'] = np.where(df_clean['Glucose'] >= 126, 1, 0)

# 3. الفئة العمرية
df_clean['Age_Group'] = pd.cut(df_clean['Age'], bins=[0, 35, 50, 120], labels=[0, 1, 2]).astype(int)

# رسم يوضح العلاقة للميزة الجديدة
plt.figure(figsize=(6, 4))
sns.barplot(data=df_clean, x='High_Glucose', y='Outcome')
plt.title('Diabetes Rate by High Glucose (>= 126)')
plt.xlabel('High_Glucose (0 = Normal, 1 = High)')
plt.ylabel('Diabetes Rate')
plt.show()

print("نسبة الإصابة حسب ارتفاع الجلوكوز:")
print((df_clean.groupby('High_Glucose')['Outcome'].mean() * 100).round(1))

**التعليق على الرسمة:**
يوضح الرسم أن المريضات اللواتي لديهن مستوى جلوكوز مرتفع (1) تصل نسبة إصابتهن بالسكري إلى 59.6%، مقارنة بـ 19.3% فقط لمن مستواهن أقل من 126 (0).

## 6. اختيار الميزات (Feature Selection) وحفظ الداتا

In [ ]:
# حساب الارتباط (Correlation) مع عمود Outcome
corr = df_clean.corr()['Outcome'].sort_values(ascending=False)

plt.figure(figsize=(10, 7))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print("معاملات الارتباط مع Outcome:")
print(corr.round(2))

**التعليق على الرسمة:**
تظهر مصفوفة الارتباط أن عمود `Glucose` هو الأقوى ارتباطاً بالنتيجة بمعامل ارتباط 0.49، يليه العمود الجديد `High_Glucose` بمعامل 0.46، ثم `BMI` بمعامل 0.31.

**جدول قرار اختيار الميزات:**

| العمود | معامل الارتباط مع Outcome | القرار | السبب |
| :--- | :---: | :---: | :--- |
| `Glucose` | 0.49 | **إبقاء** | أقوى مؤشر وعامل ارتباط بالإصابة. |
| `High_Glucose` | 0.46 | **إبقاء** | ميزة جديدة أثبتت فعالية عالية في التمييز. |
| `BMI` | 0.31 | **إبقاء** | ارتباط واضح ومؤثر طبياً. |
| `Age` | 0.24 | **إبقاء** | العمر عامل مهم في ظهور السكري. |
| `Pregnancies` | 0.22 | **إبقاء** | له ارتباط متوسط ومفيد للنموذج. |
| `Is_Obese` | 0.21 | **إبقاء** | يوضح أثر السمنة بشكل مباشر. |
| `Age_Group` | 0.21 | **إبقاء** | يسهل تقسيم الفئات العمرية. |
| `DiabetesPedigreeFunction` | 0.17 | **إبقاء** | يمثل العامل الوراثي وهو مهم طبياً. |
| `BloodPressure` | 0.17 | **إبقاء** | ارتباط مقبول ومكمل للصورة الصحية. |
| `Has_Insulin_Test` | -0.06 | **حذف** | ارتباط ضعيف جداً يقترب من الصفر. |

In [ ]:
# حذف العمود الضعيف وحفظ الملف النهائي
df_final = df_clean.drop(columns=['Has_Insulin_Test'])
df_final.to_csv('diabetes_model_ready.csv', index=False)
print("تم حفظ الملف بنجاح، والشكل النهائي:", df_final.shape)

## 7. النتائج والتوصيات

* **أبرز الاكتشافات:**
  * مستوى الجلوكوز هو العامل الأهم على الإطلاق، حيث يرتبط بقوة مع الإصابة (0.49).
  * مؤشر كتلة الجسم (BMI) والعمر وعدد مرات الحمل تأتي في المرتبة التالية كعوامل خطورة واضحة.
  * البيانات كانت تحتوي على قيم صفرية غير منطقية في 5 أعمدة وتمت معالجتها لضمان دقة التحليل.
* **التوصيات:**
  * **لإدارة العيادة:** توجيه حملات الفحص المبكر للنساء اللواتي تتجاوز أعمارهن 35 عاماً أو لديهن مؤشر كتلة جسم (BMI) أعلى من 30.
  * **للأطباء:** اعتماد قياس الجلوكوز الصائم فوق 126 كإشارة إنذار مبكر تستدعي المتابعة الفورية.

# Task 2: ملخص لطبيب العيادة

بناءً على تحليل بيانات 768 مريضة في العيادة، تبين أن هناك ثلاثة عوامل رئيسية ترتبط بشكل وثيق بالإصابة بمرض السكري؛ يتصدرها مستوى الجلوكوز في الدم كأقوى مؤشر بمعامل ارتباط بلغ 0.49 كما أوضح رسم مصفوفة الارتباط، يليه مؤشر كتلة الجسم (BMI) بمعامل ارتباط 0.31 حيث أظهر الرسم البياني ارتفاعاً ملحوظاً في أوزان المصابات مقارنة بغير المصابات، ثم يأتي العمر في المرتبة الثالثة بمعامل ارتباط 0.24. أثناء فحص السجلات، واجهنا مشكلة كبيرة تمثلت في وجود أصفار غير منطقية طبياً في عدة فحوصات، وأبرزها عمود الأنسولين الذي كان نصفه تقريباً (48.7%) مسجلاً كصفر؛ ولحل ذلك دون تشويه النتائج، قمنا بتعويض القيم المفقودة القليلة في الجلوكوز والضغط والوزن بالوسيط الحسابي، بينما استبعدنا عمود الأنسولين لكثرة النقص فيه. بناءً على هذه النتائج، نوصي العيادة بتوجيه برامج الفحص المبكر والوقائي بشكل مكثف للنساء اللواتي لديهن سمنة (مؤشر كتلة جسم 30 فأكثر) أو يتجاوزن سن الخامسة والثلاثين، مع ضرورة التأكد من تسجيل نتائج فحص الأنسولين مستقبلاً بدقة في النظام. أخيراً، من المهم جداً التوضيح أن هذه الأرقام والنتائج تُظهر "ارتباطاً" إحصائياً بين هذه العوامل والمرض، ولا تعني بالضرورة أن كل عامل منها هو "السبب" المباشر الوحيد للإصابة، لذا يبقى التشخيص الطبي السريري هو الأساس لاتخاذ القرار العلاجي.